# DistilBERT reranker for NYAY-SUTRA

Run this on Google Colab with a GPU. Do not run it in the local API environment.

It fine-tunes `distilbert-base-multilingual-cased` to score a question against a statute section, then exports an ONNX file. Point `RERANKER_PATH` in `apps/api/.env` at that file. Until the file exists, retrieval uses the embedding score only.


In [ ]:
# Colab setup. Skip on a machine that already has the packages.
# !pip install -q transformers datasets accelerate onnx onnxruntime
from pathlib import Path
import json
corpus_path = Path('app/data/seed_corpus.json')
if not corpus_path.exists():
    corpus_path = Path('/content/seed_corpus.json')
corpus = json.loads(corpus_path.read_text())
print(corpus.get('version'), len(corpus['acts']), len(corpus['sections']))


In [ ]:
sections = corpus['sections']
rows = []
for section in sections:
    passage = f"{section['title']}\n{section['body']}"
    words = [w for w in str(section.get('keywords', '')).split() if len(w) > 3][:4]
    rows.append({'question': section['title'], 'passage': passage, 'label': 1.0})
    for word in words:
        rows.append({'question': word, 'passage': passage, 'label': 1.0})
    other = next(s for s in sections if s['act_id'] != section['act_id'])
    rows.append({'question': section['title'], 'passage': f"{other['title']}\n{other['body']}", 'label': 0.0})
print(len(rows), 'pairs')


In [ ]:
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
import torch

model_name = 'distilbert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

def encode(batch):
    return tokenizer(batch['question'], batch['passage'], truncation=True, max_length=256)

ds = Dataset.from_list(rows).map(encode, batched=True)
ds = ds.rename_column('label', 'labels')
args = TrainingArguments(output_dir='/content/nyay-reranker', num_train_epochs=2, per_device_train_batch_size=8, learning_rate=2e-5, logging_steps=10)
trainer = Trainer(model=model, args=args, train_dataset=ds)
trainer.train()
trainer.save_model('/content/nyay-reranker')


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model = AutoModelForSequenceClassification.from_pretrained('/content/nyay-reranker')
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-multilingual-cased')
dummy = tokenizer('sample question', 'sample section text', return_tensors='pt')
torch.onnx.export(model, (dummy['input_ids'], dummy['attention_mask']), '/content/reranker.onnx', input_names=['input_ids', 'attention_mask'], output_names=['logits'], dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'attention_mask': {0: 'batch', 1: 'seq'}})
print('saved /content/reranker.onnx')
print('Copy it into the API host and set RERANKER_PATH to that file.')
